<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 7 — Introduction to NumPy and pandas

From Python lists to arrays and tables.

This notebook has three parts:

1. **Functions and methods.** Why we write `len(word)` but `word.upper()`. You need this to read NumPy and pandas code, which uses both styles all the time.
2. **NumPy.** Arrays of numbers in one, two, or more dimensions: vectors, matrices, and tensors.
3. **pandas.** Tables with named columns, built on top of NumPy.

**How to work through it**

- Run each code cell in order with **Shift + Enter**. Later cells use variables created in earlier ones.
- Before you run a cell, **guess what it will print**. Then check. Wrong guesses are where the learning happens.
- Cells marked **✏️ Try it** ask you to change something and run the cell again.
- The **🧩 Exercise** cells at the end of each part are for you to fill in.
- Check that the kernel in the upper right corner is set to **Python 3.12 (Conda)**.

---

## Part 1 — Functions and methods

You have already used two different ways of asking Python to do something with a value:

```python
len("abc")      # a FUNCTION: the value goes inside the parentheses
"abc".upper()   # a METHOD:   the value comes first, then a dot, then the method name
```

Both run a piece of code and give back a result. The difference is **where the code lives** and **how you call it**.

### 1.1 Functions

A **function** is a stand-alone name. It is not attached to any particular value. You call it by writing its name and passing the value(s) in parentheses:

```
function_name(value, other_value, ...)
```

Many functions are **built in** and are always available: `len`, `print`, `type`, `sum`, `max`, `min`, `sorted`, `round`, `int`, `str`.

In [ ]:
word = "banana"
numbers = [4, 1, 3]

print(len(word))        # number of characters
print(len(numbers))     # number of items
print(max(numbers))     # largest item
print(sorted(numbers))  # a new, sorted list
print(type(word))       # what kind of value is this?

Notice that `len` works on a string **and** on a list. A function like `len` is a general tool: it works on any value that "has a length".

### 1.2 Methods

A **method** is a function that belongs to a **type** of value. Every string carries the string methods with it, and every list carries the list methods. You call a method with a dot:

```
value.method_name(other_arguments)
```

The value before the dot is handed to the method automatically. You don't pass it again inside the parentheses.

In [ ]:
word = "banana"

print(word.upper())          # "BANANA"
print(word.count("a"))       # how many "a"s?  -> 3
print(word.replace("a", "o"))
print(word.startswith("ban"))

`word.count("a")` reads as: *take `word`, and run the string method `count` on it, with `"a"` as the extra argument.*

Methods are specific to their type. `upper` makes sense for text, so it is a string method. It does not exist for lists:

In [ ]:
numbers = [4, 1, 3]
numbers.append(10)      # a LIST method: add an item at the end
print(numbers)

try:
    numbers.upper()     # lists have no upper() method
except AttributeError as err:
    print("AttributeError:", err)

`AttributeError` is Python's way of saying *"this kind of value has no method (or attribute) with that name."* We catch it here with `try`/`except` only so the notebook keeps running. Normally you would just see the error.

### 1.3 So why isn't it `"abc".len()`?

A natural guess is that length should be a method, `"abc".len()`. Try it:

In [ ]:
try:
    "abc".len()
except AttributeError as err:
    print("AttributeError:", err)

Strings have no method called `len`. Python's designers made **length a function** on purpose:

- Length makes sense for many types: strings, lists, tuples, dictionaries, sets, and later NumPy arrays and pandas tables. One function, `len(x)`, works on all of them, so you don't have to remember a different method name for each type.
- Behind the scenes, each type still supplies its own code for measuring its length. That code is a special method named `__len__` (two underscores on each side, often read "dunder len"). When you call `len(x)`, Python calls `x.__len__()` for you.

In [ ]:
print(len("abc"))
print("abc".__len__())      # what len() does behind the scenes

print(len([4, 1, 3]))
print([4, 1, 3].__len__())

You should **always write `len(x)`**, never `x.__len__()`. The second form is shown only so you can see the connection.

**Rule of thumb**

| Style | Looks like | Used for |
|---|---|---|
| Function | `len(x)`, `max(x)`, `sorted(x)` | General operations that work on many types |
| Method | `x.upper()`, `x.append(5)`, `x.count("a")` | Operations that belong to one type of value |

### 1.4 How to find a value's methods

Two quick ways:

- `dir(value)` lists everything attached to a value. Names with double underscores are internal. Skip them for now.
- In Jupyter, type a value followed by a dot, e.g. `word.`, and press **Tab** to see a pop-up list.

In [ ]:
# the string methods, without the __internal__ names
[name for name in dir("abc") if not name.startswith("_")]

In [ ]:
help(str.center)   # short documentation for one method

**✏️ Try it:** In the cell below, type `word.` and press **Tab**. Pick a method you haven't used before (e.g. `title`, `find`, or `center`) and call it.

In [ ]:
word = "banana"
# type  word.  and press Tab

### 1.5 Does a method change the value, or return a new one?

This question matters a lot for NumPy and pandas.

**Strings can never be changed** ("immutable"). String methods therefore always **return a new string** and leave the original alone:

In [ ]:
word = "banana"
shout = word.upper()

print(word)    # unchanged
print(shout)   # the new string

**Lists can be changed** ("mutable"). Some list methods change the list **in place** and return `None`, meaning "nothing":

In [ ]:
numbers = [4, 1, 3]
result = numbers.sort()    # sorts the list itself

print(numbers)   # changed: [1, 3, 4]
print(result)    # None -- sort() gives nothing back

A classic beginner bug is `numbers = numbers.sort()`, which throws the list away and keeps `None`. Compare the **function** `sorted`, which leaves the original alone and returns a new list:

In [ ]:
numbers = [4, 1, 3]
new_numbers = sorted(numbers)

print(numbers)       # unchanged: [4, 1, 3]
print(new_numbers)   # [1, 3, 4]

> **Remember:** most NumPy and pandas methods behave like string methods. They **return a new result** and leave the original alone. If you want to keep the result, assign it to a name.

### 1.6 Chaining methods

When a method returns a new value, you can call another method on that result right away. This is called **chaining**, and you will see it constantly in pandas code:

In [ ]:
raw = "   Hello, World   "
clean = raw.strip().lower().replace(",", "")
print(repr(clean))

Read the chain left to right: strip the spaces → make it lowercase → remove the comma.

### 1.7 Attributes: a dot without parentheses

Sometimes the dot is followed by a name **without** parentheses. That is an **attribute**: a stored piece of information about the value, not an action. Complex numbers are a quick example:

In [ ]:
z = 3 + 4j
print(z.real)         # attribute: no parentheses
print(z.imag)         # attribute
print(z.conjugate())  # method: parentheses, it does something

In NumPy you will use attributes like `a.shape` and `a.dtype`, and in pandas `df.columns`. If you forget the parentheses on a method, Python doesn't run it. It shows you the method itself:

In [ ]:
word = "banana"
print(word.upper)     # forgot the ()
print(word.upper())   # correct

### 1.8 One more use of the dot: modules

You have also seen `math.sqrt(25)`. There the dot means *"the function `sqrt` that lives inside the module `math`."* `sqrt` is still a plain function, and `25` is passed in the parentheses.

NumPy uses **both** styles, so you will often see the same operation written two ways:

```python
np.sum(a)   # function sum() from the numpy module, array passed in
a.sum()     # method sum() of the array itself
```

Both give the same answer.

### 🧩 Exercise 1

Given `sentence = "Data tables are everywhere"`:

1. Use a **function** to print how many characters the sentence has.
2. Use a **method** to print the sentence in all capital letters.
3. Use the method `.split()` to turn the sentence into a list of words, store it as `words`, then use a **function** to print how many words there are.
4. Print `sentence` again. Did step 2 change it? Why or why not?

In [ ]:
sentence = "Data tables are everywhere"

# 1.

# 2.

# 3.

# 4.

---

## Part 2 — NumPy: arrays of numbers

**NumPy** (Numerical Python) gives Python a new kind of value, the **array** (its type is `ndarray`, "n-dimensional array"). An array holds a grid of values, usually numbers. The grid can be:

| Dimensions | Math name | Example |
|---|---|---|
| 1 | vector | daily temperatures for one week |
| 2 | matrix | grades: one row per student, one column per assignment |
| 3 or more | tensor | a color image: height × width × 3 color channels |

Almost all data-science tools (pandas, scikit-learn, matplotlib, PyTorch) are built on NumPy arrays.

By convention NumPy is imported under the short name `np`:

In [ ]:
import numpy as np

print(np.__version__)

### 2.1 Your first array

The simplest way to make an array is to pass a list to `np.array(...)`:

In [ ]:
temps = np.array([21.5, 23.0, 19.8, 25.1, 24.4])

print(temps)
print(type(temps))

Every array carries a few **attributes** (no parentheses, see §1.7):

In [ ]:
print(temps.ndim)    # number of dimensions
print(temps.shape)   # size along each dimension, as a tuple
print(temps.size)    # total number of values
print(temps.dtype)   # data type of the values: float64 = decimal number

`shape` is `(5,)`, a tuple with one entry because the array has one dimension of length 5.

### 2.2 How is an array different from a list or a tuple?

At first glance a 1-dimensional array looks like a list. There are three important differences.

**Difference 1 — arithmetic works on every element.** With a list, `*` and `+` mean *repeat* and *join*. With an array they are math, applied element by element:

In [ ]:
prices_list  = [10, 20, 30]
prices_array = np.array([10, 20, 30])

print(prices_list * 2)     # list: repeated -> [10, 20, 30, 10, 20, 30]
print(prices_array * 2)    # array: every element doubled -> [20 40 60]

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print([1, 2, 3] + [10, 20, 30])   # lists: joined end to end
print(a + b)                       # arrays: added element by element
print(a * b)                       # element by element
print(b / a)
print(a ** 2)

With a list you would need a loop or a list comprehension to do the same: `[p * 2 for p in prices_list]`. With arrays the loop is written for you, in fast compiled code. Doing math on a whole array at once is called **vectorization**.

**Difference 2 — all values have the same type.** A list can mix anything. An array stores one `dtype` for all values. If you mix types, NumPy converts everything to one common type:

In [ ]:
mixed_list = [1, 2.5, "three"]
print(mixed_list)                   # list keeps each value as it is

print(np.array([1, 2, 3]).dtype)    # all integers   -> int64
print(np.array([1, 2.5, 3]).dtype)  # one decimal    -> everything becomes float64
print(np.array([1, 2.5, "three"]))  # one string     -> everything becomes text!

The last line shows why you should keep arrays to **one kind of value**. An array of text can't do arithmetic.

**Difference 3 — fixed size.** A list grows with `.append()`. An array's size is fixed when it is created. `np.append` doesn't change the array. It builds and returns a **new** one (like the string methods in §1.5):

In [ ]:
a = np.array([1, 2, 3])
bigger = np.append(a, 4)

print(a)        # unchanged
print(bigger)   # new array

**And the tuple?** A tuple is like a list that can't be changed after it is created. It is great for a small, fixed group of values, like a pair of coordinates or an array's `shape`. Like a list, it can mix types, and `+`/`*` join and repeat it. It does no element-wise math.

| | list | tuple | NumPy array |
|---|---|---|---|
| Written as | `[1, 2, 3]` | `(1, 2, 3)` | `np.array([1, 2, 3])` |
| Can change values | yes | no | yes |
| Can change size | yes (`append`) | no | no (makes a new array) |
| Mixed types | yes | yes | no, one `dtype` |
| `x * 2` means | repeat | repeat | multiply each value |
| Typical use | general collections | small fixed records | numbers for computation |

**Speed.** Because an array stores plain numbers of one type, packed side by side in memory, NumPy can process them far faster than Python can loop over a list. Run this cell to see it (the exact times depend on your computer):

In [ ]:
import time

big_list = list(range(5_000_000))
big_array = np.arange(5_000_000)

start = time.perf_counter()
total = sum(big_list)
print(f"list  sum: {time.perf_counter() - start:.4f} seconds")

start = time.perf_counter()
total = big_array.sum()
print(f"array sum: {time.perf_counter() - start:.4f} seconds")

### 2.3 Indexing and slicing work like lists

In [ ]:
temps = np.array([21.5, 23.0, 19.8, 25.1, 24.4])

print(temps[0])      # first value
print(temps[-1])     # last value
print(temps[1:4])    # positions 1, 2, 3

### 2.4 Summaries: functions and methods

Arrays have methods for common summaries. Most also exist as NumPy functions, the two styles from §1.8:

In [ ]:
print(temps.mean())    # method
print(np.mean(temps))  # function -- same answer

print(temps.min(), temps.max())
print(temps.sum())
print(temps.argmax())  # POSITION of the largest value
print(temps.round(0))  # a new, rounded array

### 2.5 Picking values with a condition (Boolean masks)

A comparison applied to an array gives an array of `True`/`False`, one per value. That is called a **mask**. Putting the mask inside `[ ]` keeps only the `True` positions:

In [ ]:
hot = temps > 23
print(hot)           # the mask
print(temps[hot])    # only the hot days
print(temps[temps > 23])   # the same thing in one line
print(hot.sum())     # True counts as 1 -> number of hot days

**✏️ Try it:** Change the cell above to find the days *below* 22 degrees. How many are there?

### 2.6 Handy ways to create arrays

In [ ]:
print(np.arange(0, 10, 2))     # like range(): start, stop (excluded), step
print(np.linspace(0, 1, 5))    # 5 evenly spaced values from 0 to 1 (included)
print(np.zeros(4))             # four zeros
print(np.ones(3))              # three ones

### 2.7 Two dimensions: matrices

A **matrix** is a 2-dimensional array: rows and columns, like a spreadsheet of numbers. Build one from a **list of lists**. Each inner list becomes a row.

Here are the scores of 3 students (rows) on 4 homework assignments (columns):

In [ ]:
scores = np.array([
    [90, 85, 70, 100],   # student 0
    [60, 75, 80,  95],   # student 1
    [88, 92, 95,  90],   # student 2
])

print(scores)
print("ndim: ", scores.ndim)
print("shape:", scores.shape)   # (rows, columns)

Pick values with `[row, column]`. A `:` alone means "all of them" along that dimension:

In [ ]:
print(scores[0, 1])    # student 0, assignment 1  -> 85
print(scores[2])       # all of student 2's scores (row 2)
print(scores[:, 3])    # everyone's score on assignment 3 (column 3)

**Summaries along an axis.** `axis=0` goes **down the rows**, giving one result per column. `axis=1` goes **across the columns**, giving one result per row:

In [ ]:
print(scores.mean())          # overall average of all 12 scores
print(scores.mean(axis=0))    # average per ASSIGNMENT (one value per column)
print(scores.mean(axis=1))    # average per STUDENT   (one value per row)

**Element-wise math works on matrices too.** Give everyone 5 bonus points, capped at 100:

In [ ]:
curved = np.minimum(scores + 5, 100)
print(curved)

**Transpose** swaps rows and columns with the attribute `.T`:

In [ ]:
print(scores.T)
print(scores.T.shape)

**Reshape** puts the same values into a different shape (the total number of values must stay the same):

In [ ]:
flat = np.arange(12)
print(flat)
print(flat.reshape(3, 4))   # 3 rows, 4 columns
print(flat.reshape(4, 3))   # 4 rows, 3 columns

**Matrix multiplication** with `@`. This is different from `*`, which multiplies element by element.

Suppose each assignment has a weight in the final grade. Multiplying the score matrix by the weight vector gives each student's weighted total in one step. For every row, it multiplies each score by its weight and adds them up:

In [ ]:
weights = np.array([0.2, 0.2, 0.2, 0.4])   # must add up to 1

final = scores @ weights
print(final)                  # one weighted grade per student

# the same thing for student 0, done by hand:
print(90*0.2 + 85*0.2 + 70*0.2 + 100*0.4)

**✏️ Try it:** Change the weights so that assignment 3 counts for 70% and the other three for 10% each. Who benefits?

### 2.8 Three or more dimensions: tensors

Nothing stops at two dimensions. A 3-dimensional array is a stack of matrices. In machine learning, arrays with any number of dimensions are called **tensors**.

Example: unit sales for **2 stores × 3 weeks × 4 products**:

In [ ]:
sales = np.array([
    # store 0: rows = weeks, columns = products
    [[5, 2, 0, 1],
     [3, 4, 1, 0],
     [6, 1, 2, 2]],
    # store 1
    [[1, 0, 4, 3],
     [2, 2, 5, 1],
     [0, 3, 6, 2]],
])

print(sales.ndim)
print(sales.shape)   # (stores, weeks, products)

Indexing takes one position per dimension, and `axis` summaries pick which dimension to add up:

In [ ]:
print(sales[1, 2, 2])        # store 1, week 2, product 2 -> 6
print(sales[0])              # store 0's whole 3 x 4 table

print(sales.sum(axis=1))     # add up the weeks: total per store and product (2 x 4)
print(sales.sum(axis=(0, 1)))  # add up stores AND weeks: total per product

Where you will meet tensors:

- A **grayscale image** is a matrix: height × width brightness values.
- A **color image** is a 3-D tensor: height × width × 3 (red, green, blue).
- A **batch of 32 color images** for a neural network is 4-D: 32 × height × width × 3.

In [ ]:
# a tiny 2 x 2 color "image": every pixel has [red, green, blue] values from 0 to 255
image = np.array([
    [[255, 0, 0], [0, 255, 0]],      # red pixel, green pixel
    [[0, 0, 255], [255, 255, 255]],  # blue pixel, white pixel
])
print(image.shape)   # (height, width, colors)

### 🧩 Exercise 2

The array below holds the number of customers served by 3 coffee shops (rows) on 5 weekdays (columns, Monday to Friday).

1. Print the array's `shape`.
2. Print the number of customers shop 1 served on Wednesday (column 2).
3. Print the total number of customers per **shop** (hint: which `axis`?).
4. Print the average per **day**.
5. Use a mask to print all values greater than 100.
6. Each customer spends on average $6.50. Print the revenue matrix (every value × 6.5).

In [ ]:
customers = np.array([
    [ 80,  95, 110,  90, 130],
    [ 60,  70,  65,  85, 120],
    [100, 105,  98, 115, 140],
])

# 1.

# 2.

# 3.

# 4.

# 5.

# 6.

---

## Part 3 — pandas: tables with named columns

NumPy arrays are great for numbers, but real data looks more like a spreadsheet:

- Columns have **names** (`"customer_id"`, `"amount"`), not just positions.
- Different columns hold **different types**: numbers, text, dates.
- Some values are **missing**.

**pandas** handles all of this. It gives us two new types:

| Type | What it is |
|---|---|
| `Series` | one labeled column of values (a 1-D NumPy array plus labels) |
| `DataFrame` | a table: several Series side by side, sharing the same row labels |

By convention pandas is imported as `pd`:

In [ ]:
import pandas as pd

print(pd.__version__)

### 3.1 Series: a labeled column

In [ ]:
temps = pd.Series([21.5, 23.0, 19.8, 25.1, 24.4],
                  index=["Mon", "Tue", "Wed", "Thu", "Fri"])
print(temps)

The left column is the **index**, the labels. The right column holds the values. You can look values up by label, and everything you learned about arrays still works:

In [ ]:
print(temps["Wed"])        # by label
print(temps.mean())        # summary method, as with NumPy
print(temps[temps > 23])   # masks, as with NumPy
print(temps.values)        # the NumPy array underneath

### 3.2 DataFrame: a table

The easiest way to build a small table by hand is a **dictionary of lists**. Each key becomes a column name, and each list becomes the column's values. All lists must have the same length.

This is the `orders` table from the *Tabular Data* reading. Order 3's amount is unknown, so we write `None`:

In [ ]:
orders = pd.DataFrame({
    "order_id":    [1, 2, 3, 4],
    "customer_id": [10, 10, 20, 99],
    "amount":      [100.0, 50.0, None, 30.0],
    "order_date":  pd.to_datetime(["2026-01-05", "2026-01-08", "2026-01-09", "2026-01-10"]),
})

orders

Notes:

- Jupyter shows a DataFrame as a nice table when it is the **last line** of a cell (no `print` needed).
- The numbers 0–3 on the left are the **index**. pandas made them for us.
- The missing amount shows as `NaN` ("not a number"), pandas' marker for a missing number. Missing is **not** zero.

### 3.3 First look at a table

Whenever you get a new table, these are the first things to check:

In [ ]:
print(orders.shape)     # (rows, columns) -- an attribute, like NumPy
print(orders.columns)   # the column names
print(orders.dtypes)    # the type of each column

In [ ]:
orders.head(2)    # first 2 rows (default: 5)

In [ ]:
orders.info()     # columns, types, and how many values are NOT missing

In [ ]:
orders.describe()   # quick statistics for the number columns

Look at the `count` row of `describe()`: `amount` has only 3 values. The missing one is skipped.

### 3.4 Choosing columns

One column name in square brackets gives a **Series**. A **list** of names gives a smaller **DataFrame**. Notice the double brackets:

In [ ]:
orders["amount"]          # one column -> Series

In [ ]:
orders[["order_id", "amount"]]    # list of columns -> DataFrame

### 3.5 Choosing rows with a condition

Masks work just like in NumPy. Build a True/False Series from a comparison, then use it to keep rows:

In [ ]:
big = orders["amount"] > 40
print(big)

In [ ]:
orders[big]

Order 3 is not included. Its amount is unknown, so pandas can't say it is greater than 40.

To combine conditions, use `&` (and) or `|` (or), with **parentheses around each comparison**:

In [ ]:
orders[(orders["customer_id"] == 10) & (orders["amount"] >= 50)]

### 3.6 Missing values

In [ ]:
print(orders["amount"].isna())    # True where the value is missing
print(orders["amount"].sum())     # sum of the KNOWN amounts: 180

The total 180 is the sum of the **known** amounts. It does not mean order 3 was worth zero. The reading covers this in detail.

### 3.7 Sorting

`sort_values` returns a **new**, sorted table. As with string methods (§1.5), `orders` itself is unchanged until you assign the result:

In [ ]:
by_amount = orders.sort_values("amount", ascending=False)
by_amount

In [ ]:
orders    # still in the original order

### 3.8 Adding a column

Assigning to a new column name creates the column. Arithmetic on a column works element by element, like NumPy:

In [ ]:
orders["amount_with_tax"] = orders["amount"] * 1.08
orders

Missing in, missing out: the tax on an unknown amount is also unknown.

### 3.9 A first taste of grouping

"Total amount **per customer**" is one of the most common questions in business data. `groupby` splits the rows into groups by a column, and the summary method is applied to each group:

In [ ]:
orders.groupby("customer_id")["amount"].sum()

Read the chain left to right (§1.6): *take `orders` → group by `customer_id` → look at `amount` → sum each group.*

(Customer 20's only amount is missing, so pandas shows a total of `0.0`. The reading explains why that can be misleading and how to fix it. We'll come back to grouping in the next notebooks.)

### 3.10 Reading a table from a CSV file

In practice you rarely type tables by hand. You load them from files. `pd.read_csv` reads a CSV file into a DataFrame. Here we write a small file first so the example is self-contained:

In [ ]:
from pathlib import Path

csv_text = '''product,category,price,stock
Coffee,Beverage,4.50,120
Tea,Beverage,3.25,80
Bagel,Food,2.75,40
Muffin,Food,3.00,0
Juice,Beverage,5.00,35
'''
Path("products.csv").write_text(csv_text, encoding="utf-8")

products = pd.read_csv("products.csv")
products

In [ ]:
products.dtypes    # pandas guessed the type of each column

**✏️ Try it:** Show only the products in the `"Beverage"` category, sorted by price from cheapest to most expensive. (Hint: first a mask, then `.sort_values("price")`.)

### 🧩 Exercise 3

Use the `products` table from §3.10.

1. Print how many rows and columns it has.
2. Show only the `product` and `price` columns.
3. Show the products that are **out of stock** (`stock == 0`).
4. Add a column `stock_value` = `price * stock`, then show the table.
5. What is the total `stock_value` of all products?
6. Use `groupby` to compute the average price per `category`.
7. **Bonus:** Which product has the highest `stock_value`? (Hint: sort, then `head(1)`.)

In [ ]:
# 1.

# 2.

# 3.

# 4.

# 5.

# 6.

# 7.

---

## Summary

**Functions and methods**

- A **function** stands alone: `len(x)`, `sorted(x)`, `np.mean(a)`.
- A **method** belongs to a type and is called with a dot: `x.upper()`, `a.mean()`, `df.sort_values("amount")`.
- An **attribute** is stored information, with no parentheses: `a.shape`, `df.columns`.
- `len(x)` is a function so that one name works for every type. It calls the type's own `__len__` behind the scenes.
- Most NumPy and pandas methods **return a new result**. Assign it to a name if you want to keep it.

**NumPy**

- `np.array(...)` makes an array: one `dtype`, fixed size, fast element-wise math.
- 1-D = vector, 2-D = matrix, 3-D and more = tensor. `shape` tells you the size of each dimension.
- `axis=0` summarizes down the rows (one result per column), `axis=1` across the columns (one result per row).
- `*` multiplies element by element. `@` is matrix multiplication.

**pandas**

- A `Series` is one labeled column. A `DataFrame` is a table of named columns.
- First look: `shape`, `columns`, `dtypes`, `head()`, `info()`, `describe()`.
- `df["col"]` → a Series. `df[["a", "b"]]` → a DataFrame. `df[mask]` → the matching rows.
- Missing values (`NaN`) are **unknown**, not zero.
- `groupby(...)` + a summary answers "per customer / per category" questions.

**Next:** `02_Northwind_SQL_Pandas.ipynb` works with several related tables and joins them, in both SQL and pandas.